### Tools in Langchain

1 How to create Tools

2 How to use built-in tools and toolkits

3 How to use chat models to call tools

4 how to pass tool outputs to chat models

This @tool decorator is the simplest way to define a custom tool. The decorator uses the function name as the tool name by default, but this can be overridden by passing a string as the first argument. Additionally, the decorator will use the function's docstring as the tool's description so a docstring Must be provided.

In [1]:
## @tool decorator

from langchain_core.tools import tool

@tool
def division(a:int,b:int)->int:
    """Divide 2 numbers"""
    return a/b

In [2]:
print(division.name)
print(division.description)
print(division.args)

division
Divide 2 numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [3]:
## async implementation

from langchain_core.tools import tool
@tool
async def amultiply(a: int, b:int)-> int:
    """Multiply 2 numbers"""
    return a * b

In [4]:
from typing import Annotated, List

@tool
def multiply_by_max(
    a: Annotated[int, "A value"],
    b: Annotated[List[int],"B Value" ]
)-> int:
    """Mulitply a by the maximum b"""
    return a* max(b)


In [5]:
print(multiply_by_max.args)

{'a': {'description': 'A value', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'B Value', 'items': {'type': 'integer'}, 'title': 'B', 'type': 'array'}}


### Structured Tool
The StructuredTool.from_function class method provides a bit more cofigurability than the @tool decorator, without requiring much additional code.

In [6]:
from langchain_core.tools import StructuredTool

def multiply(a: int, b:int)-> int:
    """multiply 2 numbers"""
    return a * b

async def amultiply(a: int, b:int)-> int:
    """multiply 2 numbers"""
    return a * b

calculator=StructuredTool.from_function(func=multiply, coroutine=amultiply)

print(calculator.invoke({'a':2,'b':3}))
print(await calculator.ainvoke({'a':2,'b':5}))

6
10


### Inbuilt Tools

wikipedia Integration

In [11]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper=WikipediaAPIWrapper(top_k_results=5,doc_content_chars_max=500)
tool= WikipediaQueryRun(api_wrapper=api_wrapper)

print(tool.invoke({"query":"langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.



Page: Vector database
Summary: A vector database, vector store or vector search engine is a database that stores and retrieves embeddings of data in v


In [13]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [15]:
from langchain_tavily import TavilySearch

tool = TavilySearch(
    max_results=5,
    topic="general"
    # include_answer=False,
    #include_raw_content=False,
    #include_images=False,
    #include_image_descriptions=False,
    #search_depth="basic",
    #time_range="day",
    #include_domains=None,
    #exclude_domains=None
)

In [16]:
tool.invoke("What is the recent Ai news?")

{'query': 'What is the recent Ai news?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.youtube.com/@AINewsOfficial',
   'title': 'AI News - YouTube',
   'content': 'AI News delivers the latest breakthroughs in artificial intelligence, machine learning, deep learning, brain computer interface, robotics, and other future',
   'score': 0.6659544,
   'raw_content': None},
  {'url': 'https://www.wsj.com/tech/ai',
   'title': 'Artificial Intelligence - Latest AI News and Analysis - WSJ.com',
   'content': "Meta Announces New AI Model in Major Test of Company's Ambitions · The disappointment of last model's release more than a year ago led to an expensive overhaul",
   'score': 0.6633424,
   'raw_content': None},
  {'url': 'https://www.artificialintelligence-news.com/',
   'title': 'AI News | Latest News | Insights Powering AI-Driven Business Growth',
   'content': 'Secure governance accelerates financial AI revenue growth · Thailand becomes o

In [18]:
from langchain_community.utilities import ArxivAPIWrapper

tool=ArxivAPIWrapper()
tool.run("1706.03762")

'Published: 2023-08-02\nTitle: Attention Is All You Need\nAuthors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin\nSummary: The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task, improving over the existing best results, including ensembles by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model establishes a new 

## Call Tool With LLM Model

In [3]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=100)
wiki_tool= WikipediaQueryRun(api_wrapper=api_wrapper)

print(wiki_tool.invoke({"query":"langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of 


In [6]:
from langchain_tavily import TavilySearch

tavily_tool = TavilySearch(
    max_results=5,
    topic="general"
    # include_answer=False,
    #include_raw_content=False,
    #include_images=False,
    #include_image_descriptions=False,
    #search_depth="basic",
    #time_range="day",
    #include_domains=None,
    #exclude_domains=None
)

In [7]:
from langchain_core.tools import tool

@tool
def add(a:int, b:int)->int:
    """Add Two numbers"""
    return a + b

@tool
def multiply(a:int, b:int)->int:
    """Multiply Two numbers"""
    return a * b  

In [8]:
tools=[wiki_tool,add,multiply,tavily_tool]

In [9]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'e:\\AgenticAI\\.venv\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=100)),
 StructuredTool(name='add', description='Add Two numbers', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x000001E623920EE0>),
 StructuredTool(name='multiply', description='Multiply Two numbers', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x000001E621283B50>),
 TavilySearch(max_results=5, topic='general', api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'), api_base_url=None))]

In [10]:
from langchain.chat_models import init_chat_model
llm= init_chat_model("qwen/qwen3-32b",model_provider="groq")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E623BA5390>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E623BA52A0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [11]:
llm_with_tools=llm.bind_tools(tools)
llm_with_tools

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E623BA5390>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E623BA52A0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'wikipedia', 'description': 'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.', 'parameters': {'properties': {'query': {'description': 'query to look up on wikipedia', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}}}, {'type': 'function

In [19]:
from langchain_core.messages import HumanMessage
query="What is 2 * 3"
messages = [HumanMessage(query)]
response= llm_with_tools.invoke(query)
print(response)

content='' additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user is asking "What is 2 * 3". That\'s a multiplication problem. I need to figure out which tool to use here. Looking at the available functions, there\'s one called multiply. The description says it\'s for multiplying two numbers. The parameters are a and b, both integers.\n\nSo, the user\'s question is straightforward. They want to multiply 2 by 3. I should check if the multiply function is the right choice. Since the inputs are integers and the operation is multiplication, yes, that\'s exactly what the multiply function does. \n\nI don\'t need to use the other tools like wikipedia or tavily_search because this is a simple math problem. The add function is for addition, which isn\'t needed here. So, the correct tool is multiply with a=2 and b=3. \n\nJust need to make sure the parameters are correctly formatted as integers. The function requires both a and b, so I\'ll include them. No other parameters are neces

In [20]:
response.tool_calls

[{'name': 'multiply',
  'args': {'a': 2, 'b': 3},
  'id': '6gnys37kr',
  'type': 'tool_call'}]

In [22]:
for tool_call in response.tool_calls:
    selected_tool = { "add": add, "multiply": multiply,"wikipedia":wiki_tool,"tavily_search":tavily_tool}[tool_call["name"].lower()]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

messages

[ToolMessage(content='6', name='multiply', tool_call_id='6gnys37kr'),
 ToolMessage(content='6', name='multiply', tool_call_id='6gnys37kr')]

In [23]:
llm_with_tools.invoke(messages)

AIMessage(content='The sum of 3 and 3 is 6.\n\n', additional_kwargs={'reasoning_content': 'Okay, let me try to figure this out. The user asked for the sum of 3 and 3. I remember there\'s a function called \'add\' that takes two numbers. So I should call that function with a=3 and b=3. Let me check the parameters again to make sure. The \'add\' function requires integers a and b. Yep, that\'s right. So the tool call would be name: add, arguments: { "a": 3, "b": 3 }. Then the result should be 6. That\'s straightforward. I don\'t think I need to use any other tools here. Just the \'add\' function.\n', 'tool_calls': [{'id': 'fc727agwr', 'function': {'arguments': '{"a":3,"b":3}', 'name': 'add'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 178, 'prompt_tokens': 1938, 'total_tokens': 2116, 'completion_time': 0.321170049, 'completion_tokens_details': {'reasoning_tokens': 137}, 'prompt_time': 0.082346972, 'prompt_tokens_details': None, 'queue_time': 0.16088371

In [24]:
from langchain_core.messages import HumanMessage

query = "What is langchain and what is 5*15?"
messages =[HumanMessage(query)]

ai_msg = llm_with_tools.invoke(messages)
ai_msg

AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking two things here: what LangChain is and the result of 5 multiplied by 15. Let me break this down.\n\nFirst, for the LangChain part. I need to provide a clear definition. Since LangChain is a framework related to large language models (LLMs), I should explain its purpose, maybe mention its key features like prompting, memory, agents, and integration with LLMs. The user might be looking for a concise overview, so I should keep it straightforward but informative. The Wikipedia function could be useful here because LangChain has a detailed page there. Using the Wikipedia tool with the query "LangChain" should give me the necessary information to summarize.\n\nNext, the math part: 5*15. That\'s a simple multiplication. The user probably wants a quick answer. Since there\'s a multiply function available, I can use that. The parameters would be a=5 and b=15. I don\'t need to use the search functions here be

In [25]:
ai_msg.tool_calls

[{'name': 'wikipedia',
  'args': {'query': 'LangChain'},
  'id': '73axs95fd',
  'type': 'tool_call'},
 {'name': 'multiply',
  'args': {'a': 5, 'b': 15},
  'id': '0etxjz3nz',
  'type': 'tool_call'}]

In [26]:
for tool_call in ai_msg.tool_calls:
    selected_tool = { "add": add, "multiply": multiply,"wikipedia":wiki_tool,"tavily_search":tavily_tool}[tool_call["name"].lower()]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

messages

[HumanMessage(content='What is langchain and what is 5*15?', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='Page: LangChain\nSummary: LangChain is a software framework that helps facilitate the integration of ', name='wikipedia', tool_call_id='73axs95fd'),
 ToolMessage(content='75', name='multiply', tool_call_id='0etxjz3nz')]

In [27]:
llm_with_tools.invoke(messages)

AIMessage(content='LangChain is a software framework designed to help developers integrate and manage various components of AI applications, such as language models, data sources, and APIs. It provides tools for tasks like prompt management, model orchestration, and data processing, streamlining the development of AI-driven workflows. \n\nFor the calculation:  \n$ 5 \\times 15 = 75 $.', additional_kwargs={'reasoning_content': 'Okay, let\'s break down the user\'s question. They asked two things: what LangChain is and what 5 multiplied by 15 is. \n\nFirst, for the LangChain part, I need to use the Wikipedia tool. The user probably wants a clear, concise definition. I\'ll search Wikipedia for "LangChain" to get the most accurate and up-to-date information. The summary from the tool response mentions it\'s a software framework for integrating AI components. I should present that in simple terms.\n\nNext, the math problem: 5 times 15. That\'s straightforward. Using the multiply function mak

In [28]:
response=llm_with_tools.invoke(messages)

In [29]:
response.content

'1. **LangChain**:  \nLangChain is a software framework designed to streamline the integration of language models (LLMs) with external tools, data sources, and application logic. It provides components like prompts, memory management, agents, and chains to build scalable AI applications. For example, it can help create chatbots that access databases or automate workflows using LLMs.\n\n2. **5 × 15**:  \nThe result of multiplying 5 by 15 is **75**.'